# ForecastIQ — Notebook 3C: Scaling Global Direct and Hurdle LightGBM

**Research question:** Does training on more product-store series improve forecasts on the **same fixed holdout population**?

Default experiments use 400, 2,000, 5,000 training series; optionally 10,000 and all 30,490. A fixed **evaluation cohort** (default 400 series) is drawn from the full eligible population and excluded from *all* ML training-cohort choices. The ML models can forecast unseen series from their history using store/category product descriptors (not series-specific codes). The 28-day test is the same final M5 validation-file window used previously. **These are new-series transfer/generalization experiments, not the same estimand as Notebook 3A**, which trained with the evaluated series included. Statistical benchmarks for the fixed evaluation cohort can come from Notebook 2B.

**Hardware caution:** build training examples in chunks / limit training origins; full size is expensive. Start with `TRAINING_SIZES=[400,2000]`, `MAX_ORIGINS=6`. We use time boundaries and no test labels for training; tuning remains fixed in this initial scaling study.

**Corrected protocol:** training-size and hyperparameter choices require historical validation; the previously explored final 28 days are descriptive only. Seen-series and unseen-series results must be exported separately.

**Corrected protocol:** training-size and hyperparameter choices require historical validation; the previously explored final 28 days are descriptive only. Seen-series and unseen-series results must be exported separately.

**Corrected protocol:** training-size and hyperparameter choices require historical validation; the previously explored final 28 days are descriptive only. Seen-series and unseen-series results must be exported separately.


In [6]:
# Shared scientifically validated ForecastIQ framework
import sys
from pathlib import Path
_project = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(_project) not in sys.path:
    sys.path.insert(0, str(_project))
from src.experiments.config import ExperimentConfig, save_manifest
EXPERIMENT_CONFIG = ExperimentConfig()
EXPERIMENT_NAME = '03C_large_scale_lightgbm_hurdle'
MANIFEST = save_manifest(_project / 'results' / 'manifests' / (EXPERIMENT_NAME + '.json'), EXPERIMENT_NAME, EXPERIMENT_CONFIG, sources=[_project / 'data' / 'raw' / 'sales_train_validation.csv'])
print('Experiment manifest:', MANIFEST['experiment_id'])


Experiment manifest: 03C_large_scale_lightgbm_hurdle-20261101T103237Z


## 1 — Load full exports and define fixed evaluation cohort

In [7]:
from pathlib import Path
import numpy as np,pandas as pd,time,gc
from IPython.display import display
import lightgbm as lgb
PROJECT=Path.cwd()
if not (PROJECT/'data').exists() and (PROJECT.parent/'data').exists():PROJECT=PROJECT.parent
OUT=PROJECT/'data'/'processed';OUT.mkdir(parents=True,exist_ok=True)
H=28;SEED=42;MIN_HISTORY=112
EVAL_N=400
TRAINING_SIZES=[400,2000,5000,10000] # maximum eligible size is appended below
MAX_ORIGINS=8 # historical origin windows per stage. Increase after smoke test.
STEP=28;UNDER=5.;OVER=1.
sales=pd.read_csv(OUT/'m5_full_sales.csv')
meta=pd.read_csv(OUT/'m5_full_metadata.csv')
meta=meta[['series_id','demand_type']]
full=sales.merge(meta,on='series_id',validate='one_to_one')
days=sorted([c for c in full if c.startswith('d_') and c[2:].isdigit()],key=lambda c:int(c[2:]))
Y=full[days].to_numpy(dtype=np.float32)
assert np.isfinite(Y).all() and (Y>=0).all()
N,T=Y.shape;TEST=T-H;VALIDATION=TEST-H
# Balanced evaluation by label for diagnostic coverage, disjoint from training by series.
rng=np.random.default_rng(SEED)
eval_pos=[]
for cat in ['Smooth','Erratic','Intermittent','Lumpy']:
    eligible=np.flatnonzero(full.demand_type.to_numpy()==cat)
    eval_pos.extend(rng.choice(eligible,size=min(EVAL_N//4,len(eligible)),replace=False).tolist())
eval_pos=np.array(sorted(eval_pos))
train_pool=np.setdiff1d(np.arange(N),eval_pos)
permutation=rng.permutation(train_pool)
TRAINING_SIZES=sorted(set([s for s in TRAINING_SIZES if s<=len(permutation)]+[len(permutation)]))
assert len(TRAINING_SIZES)>0
assert len(set(eval_pos)&set(permutation))==0
print('Full:',N,'fixed evaluation:',len(eval_pos),'available training:',len(permutation),'train sizes:',TRAINING_SIZES)

Full: 30490 fixed evaluation: 400 available training: 30090 train sizes: [400, 2000, 5000, 10000, 30090]


## 2 — Past-only feature engineering and training origins

Each supervised row predicts a future day's sales from a **frozen historic snapshot**, so nothing from that future 28-day block enters its features. Evaluation series do not appear in the training pool (product `item_id` can still repeat at other stores—this is a **new series**, not a completely unseen item experiment).

For efficiency, train on a bounded number of past origin windows and use compact `float32` / categorical columns.

In [8]:
from importlib import reload
import src.features.feature_engineering as feature_engineering
feature_engineering=reload(feature_engineering)

calendar_path=PROJECT/'data'/'raw'/'calendar.csv'
if calendar_path.exists():
    cal=pd.read_csv(calendar_path,usecols=['d','date']).drop_duplicates('d').set_index('d')
    dates=pd.DatetimeIndex(pd.to_datetime(cal.reindex(days)['date']))
    assert not dates.isna().any()
else:
    dates=pd.date_range('2011-01-29',periods=T,freq='D')

def features_for(ids,origin):
    hist=Y[ids,:origin]
    assert origin>=MIN_HISTORY and origin+H<=T
    snap=feature_engineering.historical_snapshot(hist)
    n=len(ids)
    data={'horizon':np.tile(np.arange(1,H+1,dtype=np.int16),n),
          'store':np.repeat(full.store_id.iloc[ids].to_numpy(),H),
          'department':np.repeat(full.item_id.iloc[ids].astype(str).str.split('_').str[:2].str.join('_').to_numpy(),H)}
    for key,value in snap.items():
        if key=='last_28_vs_prev_28':
            continue
        output_key=key.replace('zero_ratio_','zero_')
        data[output_key]=np.repeat(value,H)
    future=dates[origin:origin+H]
    data['weekday']=np.tile(future.dayofweek.to_numpy(dtype=np.int8),n)
    data['month']=np.tile(future.month.to_numpy(dtype=np.int8),n)
    return pd.DataFrame(data)

def prep(frame):
    for column in ['store','department']:
        frame[column]=frame[column].astype('category')
    return frame

FEATURES=['horizon','store','department','lag_1','lag_7','lag_14','lag_28','lag_56',
          'mean_7','zero_7','std_7','mean_28','zero_28','std_28','mean_56','zero_56','std_56','weekday','month']
assert TEST>MIN_HISTORY+H*4
all_origins=list(range(MIN_HISTORY,VALIDATION-H+1,STEP))
train_origins=all_origins[-MAX_ORIGINS:] if MAX_ORIGINS is not None else all_origins
assert max(train_origins)+H<=VALIDATION
print('Training origins:',train_origins)
check=features_for(eval_pos[:2],TEST);backup=Y[eval_pos[0],TEST]
Y[eval_pos[0],TEST]=backup+999
pd.testing.assert_frame_equal(check,features_for(eval_pos[:2],TEST))
Y[eval_pos[0],TEST]=backup
print('Future-sales leakage smoke test PASSED')


Training origins: [1624, 1652, 1680, 1708, 1736, 1764, 1792, 1820]
Future-sales leakage smoke test PASSED


## 3 — Train Direct + Hurdle LightGBM on increasing training series

Categorical features do not require `series_code`, so evaluation on entirely withheld series is possible. Positive-size regression is fitted only on positive targets. Unseen store/department categories are set to missing rather than receiving inconsistent category codes. Training sizes are nested via a fixed permutation.

In [9]:
from src.forecasting.lightgbm_hurdle import fit_predict_hurdle_and_direct_frames
from src.evaluation.metrics import mae, wape, mase, rmsse, mean_signed_bias, inventory_cost_proxy

def make_training(ids,forecast_origin):
    eligible=list(range(MIN_HISTORY,forecast_origin-H+1,STEP))
    origins=eligible[-MAX_ORIGINS:] if MAX_ORIGINS is not None else eligible
    assert max(origins)+H<=forecast_origin
    parts=[features_for(ids,o) for o in origins]
    targets=[Y[ids,o:o+H].ravel() for o in origins]
    return prep(pd.concat(parts,ignore_index=True))[FEATURES],np.concatenate(targets)

def evaluation_frame(train_X,origin,ids=eval_pos):
    frame=prep(features_for(ids,origin))[FEATURES]
    for column in ['store','department']:
        frame[column]=pd.Categorical(frame[column],categories=train_X[column].cat.categories)
    return frame

def train_predict(ntrain,forecast_origin):
    idx=permutation[:ntrain];start=time.time()
    X,targets=make_training(idx,forecast_origin);X_future=evaluation_frame(X,forecast_origin)
    params=dict(n_estimators=200,learning_rate=.05,num_leaves=31,min_child_samples=80,
                reg_lambda=1.,random_state=SEED,n_jobs=-1,verbosity=-1)
    result=fit_predict_hurdle_and_direct_frames(X,targets,X_future,params=params,
                                                categorical_features=['store','department'])
    direct=result['direct'].reshape(len(eval_pos),H);hurdle=result['hurdle'].reshape(len(eval_pos),H)
    elapsed=time.time()-start
    del X,X_future,targets,result;gc.collect()
    return direct,hurdle,elapsed

truth=Y[eval_pos,VALIDATION:VALIDATION+H]
results=[];forecasts={}
for ntrain in TRAINING_SIZES:
    print('Training on',ntrain,'series...')
    direct,hurdle,elapsed=train_predict(ntrain,VALIDATION)
    for model,prediction in [('Direct',direct),('Hurdle',hurdle)]:
        results.append({'training_series':ntrain,'model':model,'eval_series':len(eval_pos),
                        'MAE':mae(truth,prediction),'WAPE_pct':100*wape(truth,prediction),
                        'bias':mean_signed_bias(truth,prediction),
                        'cost_proxy':inventory_cost_proxy(truth,prediction,UNDER,OVER),
                        'seconds_for_both_models':elapsed})
        forecasts[(ntrain,model)]=prediction
summary=pd.DataFrame(results);display(summary.round(4))

selected_row=summary.sort_values(['MAE','training_series','model']).iloc[0]
selected_n=int(selected_row.training_series);selected_model=str(selected_row.model)
print('Validation-selected final configuration:',selected_n,selected_model)
final_direct,final_hurdle,final_elapsed=train_predict(selected_n,TEST)
final_truth=Y[eval_pos,TEST:TEST+H]
final_pred=final_direct if selected_model=='Direct' else final_hurdle
per_series=pd.DataFrame({'MASE':[mase(a,p,t) for a,p,t in zip(final_truth,final_pred,Y[eval_pos,:TEST])],
                         'RMSSE':[rmsse(a,p,t) for a,p,t in zip(final_truth,final_pred,Y[eval_pos,:TEST])]})
final_summary=pd.DataFrame([{'scenario':'unseen_product_store_combination','training_series':selected_n,
    'model':selected_model,'eval_series':len(eval_pos),'MAE':mae(final_truth,final_pred),
    'WAPE_pct':100*wape(final_truth,final_pred),'MASE':per_series.MASE.mean(),'RMSSE':per_series.RMSSE.mean(),
    'bias':mean_signed_bias(final_truth,final_pred),'cost_proxy':inventory_cost_proxy(final_truth,final_pred,UNDER,OVER),
    'seconds_for_both_models':final_elapsed}])
display(final_summary.round(4));final_summary.to_csv(OUT/'phase3c_validation_selected_final_test.csv',index=False)


Training on 400 series...
Training on 2000 series...
Training on 5000 series...
Training on 10000 series...
Training on 30090 series...


,training_series,model,eval_series,MAE,WAPE_pct,bias,cost_proxy,seconds_for_both_models
0,400,Direct,400,2.1480,60.4308,-0.0327,72906.0412,1.5685
1,400,Hurdle,400,2.2206,62.4737,0.0313,73910.2156,1.5685
2,2000,Direct,400,2.0700,58.2379,0.0592,68226.8256,4.2863
3,2000,Hurdle,400,2.0780,58.4618,0.0508,68682.3890,4.2863
4,5000,Direct,400,2.0356,57.2679,-0.0525,69570.0514,10.5398
5,5000,Hurdle,400,2.0470,57.5908,0.0415,67850.9476,10.5398
6,10000,Direct,400,2.0442,57.5112,0.0481,67607.6067,23.1600
7,10000,Hurdle,400,2.0599,57.9538,0.0346,68438.2350,23.1600
8,30090,Direct,400,2.0379,57.3331,0.0369,67646.8366,63.6462
9,30090,Hurdle,400,2.0432,57.4821,0.0379,67801.3416,63.6462


Validation-selected final configuration: 5000 Direct


,scenario,training_series,model,eval_series,MAE,WAPE_pct,MASE,RMSSE,bias,cost_proxy,seconds_for_both_models
0,unseen_product_store_combination,5000,Direct,400,2.0387,59.2454,1.4986,0.8594,0.0739,66846.8649,14.2143


## 4 — Evaluate by demand type and export same-panel forecasts

These are **unseen-series** test results; expect differences from pilot results, because Notebook 3A/B trained on the 400 evaluated series themselves. Because our evaluation sample is demand-balanced, its overall MAE is not a natural-population estimate; repeat later with representative full-population evaluation.

In [ ]:
diagnostics=[]
for (ntrain,name),pred in forecasts.items():
    for cat in ['Smooth','Erratic','Intermittent','Lumpy']:
        mask=(full.demand_type.iloc[eval_pos].to_numpy()==cat)
        if not mask.any():continue
        diff=pred[mask]-truth[mask]
        diagnostics.append({'training_series':ntrain,'model':name,'demand_type':cat,'series':int(mask.sum()),
                            'MAE':np.abs(diff).mean(),'bias':diff.mean()})
by_type=pd.DataFrame(diagnostics)
display(by_type.round(4))
summary.to_csv(OUT/'phase3c_scaling_validation.csv',index=False)
by_type.to_csv(OUT/'phase3c_scaling_by_type.csv',index=False)
print('Saved outputs',OUT)

## 5 — Conclusions and next improvements

- A fixed and **disjoint** evaluation panel makes training-size comparisons meaningful for unseen product-store combinations.
- For **existing-series future forecasting**, repeat with training series allowed in the training pool (but never future outcomes): that is a separate experiment and aligns with Notebook 3A/B.
- For large data, use smaller `MAX_ORIGINS`, 2,000 series initially, monitor RAM and training time.
- Expand to 10,000 and all eligible training series only after the pilot works.
- Later: add multi-origin evaluation, tuning and product/store-aware paired uncertainty intervals.
- Do not claim the current full-scale study has already been executed: results are produced when run locally.